# Scikit-learn API Patterns
## `fit` · `predict` · `transform` · `Pipeline` · `ColumnTransformer`

---
This notebook covers the **core scikit-learn API** with real, runnable examples.

### Roadmap
| # | Topic |
|---|-------|
| 1 | Setup & Quick Intro |
| 2 | `fit` and `predict` – Classification |
| 3 | `fit` and `predict` – Regression |
| 4 | `fit` and `transform` – Preprocessing |
| 5 | `fit_transform` shorthand |
| 6 | `Pipeline` – chaining steps |
| 7 | `ColumnTransformer` – mixed-type data |
| 8 | Full end-to-end example (Titanic-style) |
| 9 | Evaluation & Cross-validation |

---
## 1. Setup

In [1]:
import sys
!{sys.executable} -m pip install scikit-learn pandas numpy --quiet

import numpy as np
import pandas as pd
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"numpy version:        {np.__version__}")
print(f"pandas version:       {pd.__version__}")


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip
scikit-learn version: 1.8.0
numpy version:        2.4.4
pandas version:       3.0.3


---
## 2. The Three Core Methods

Almost every scikit-learn estimator follows the same API:

| Method | When to use | What it returns |
|--------|------------|------------------|
| `fit(X, y)` | **Training** — learns from data | `self` (the fitted estimator) |
| `predict(X)` | **Inference** — makes predictions | array of predicted labels/values |
| `transform(X)` | **Preprocessing** — converts data | transformed array |
| `fit_transform(X)` | `fit` then `transform` in one step | transformed array |
| `predict_proba(X)` | Probabilistic output (classifiers) | array of class probabilities |

> **Mental model:** `fit` is the *learning* phase. `predict`/`transform` is the *application* phase.  
> You **always** call `fit` first!

---
## 3. `fit` and `predict` – Classification (Iris Dataset)

**Task:** Classify iris flowers into 3 species using petal/sepal measurements.

In [11]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# ── Load data ──────────────────────────────────────────────────────────
iris = load_iris()
X, y = iris.data, iris.target
print(f"Feature matrix shape: {X.shape}  (samples × features)")
print(f"Target vector shape:  {y.shape}")
print(f"Classes: {iris.target_names}")

Feature matrix shape: (150, 4)  (samples × features)
Target vector shape:  (150,)
Classes: ['setosa' 'versicolor' 'virginica']


In [3]:
# ── Train / test split ─────────────────────────────────────────────────
# random_state=42 → reproducible results
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

Training samples: 120, Test samples: 30


In [4]:
# ── Step 1: Create estimator ───────────────────────────────────────────
knn = KNeighborsClassifier(n_neighbors=3)

# ── Step 2: fit() — train the model ───────────────────────────────────
knn.fit(X_train, y_train)   # Returns the estimator itself
print("Model trained!")
print(knn)                   # Shows the estimator with its parameters

Model trained!
KNeighborsClassifier(n_neighbors=3)


In [5]:
# ── Step 3: predict() — apply to new data ─────────────────────────────
y_pred = knn.predict(X_test)
print("Predicted labels:", y_pred)
print("Actual labels:   ", y_test)

acc = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {acc:.2%}")

Predicted labels: [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 1 1 0 2 0]
Actual labels:    [0 2 1 1 0 1 0 0 2 1 2 2 2 1 0 0 0 1 1 2 0 2 1 2 2 1 1 0 2 0]

Accuracy: 100.00%


In [6]:
# ── Detailed report ────────────────────────────────────────────────────
print(classification_report(y_test, y_pred, target_names=iris.target_names))

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00        10
   virginica       1.00      1.00      1.00        10

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



In [7]:
# ── predict_proba(): probability for each class ────────────────────────
proba = knn.predict_proba(X_test[:5])
print("Class probabilities for first 5 test samples:")
df_proba = pd.DataFrame(proba, columns=iris.target_names)
df_proba['predicted'] = iris.target_names[y_pred[:5]]
print(df_proba)

Class probabilities for first 5 test samples:
   setosa  versicolor  virginica   predicted
0     1.0    0.000000   0.000000      setosa
1     0.0    0.333333   0.666667   virginica
2     0.0    1.000000   0.000000  versicolor
3     0.0    1.000000   0.000000  versicolor
4     1.0    0.000000   0.000000      setosa


---
## 4. `fit` and `predict` – Regression (Boston Housing / California Housing)

**Task:** Predict median house price from housing features.

In [12]:
# ── SSL Fix: patches urllib at the opener level (works with sklearn) ───
import ssl
import urllib.request
_ctx = ssl._create_unverified_context()
urllib.request.install_opener(
    urllib.request.build_opener(
        urllib.request.HTTPSHandler(context=_ctx)
    )
)
print("✅ urllib opener patched — SSL verification disabled for this session")

✅ urllib opener patched — SSL verification disabled for this session


In [13]:
# import ssl
# ssl._create_default_https_context = ssl._create_unverified_context
# print("SSL verification disabled ✅")
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

housing = fetch_california_housing()
X, y = housing.data, housing.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Features: {housing.feature_names}")
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Features: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Train: (16512, 8), Test: (4128, 8)


In [14]:
# ── Linear Regression ─────────────────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train, y_train)          # fit
y_pred_lr = lr.predict(X_test)    # predict

mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr  = r2_score(y_test, y_pred_lr)
print(f"Linear Regression → MSE: {mse_lr:.3f}  R²: {r2_lr:.3f}")

# ── Ridge Regression (adds L2 regularisation) ─────────────────────────
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)

mse_ridge = mean_squared_error(y_test, y_pred_ridge)
r2_ridge  = r2_score(y_test, y_pred_ridge)
print(f"Ridge Regression  → MSE: {mse_ridge:.3f}  R²: {r2_ridge:.3f}")

Linear Regression → MSE: 0.556  R²: 0.576
Ridge Regression  → MSE: 0.556  R²: 0.576


In [16]:
import pandas as pd
# ── Inspect learned parameters after fit ──────────────────────────────
coef_df = pd.DataFrame({
    'feature': housing.feature_names,
    'coefficient': lr.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print("Feature coefficients (by magnitude):")
print(coef_df.to_string(index=False))
print(f"\nIntercept: {lr.intercept_:.4f}")

Feature coefficients (by magnitude):
   feature  coefficient
 AveBedrms     0.783145
    MedInc     0.448675
 Longitude    -0.433708
  Latitude    -0.419792
  AveRooms    -0.123323
  HouseAge     0.009724
  AveOccup    -0.003526
Population    -0.000002

Intercept: -37.0233


---
## 5. `fit` and `transform` – Preprocessing Transformers

Transformers learn statistics from training data (`fit`), then apply them (`transform`).  
**Key rule:** Fit on training data **only** — never on test data (avoids data leakage).

In [18]:
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# Toy data
data = np.array([[10, 200],
                 [20, 400],
                 [30, 100],
                 [40, 300]])

# ── StandardScaler: (x - mean) / std  ─────────────────────────────────
scaler = StandardScaler()
scaler.fit(data)             # learns mean & std from this data
print("Learned mean:", scaler.mean_)
print("Learned std: ", scaler.scale_)

data_scaled = scaler.transform(data)
print("\nOriginal data:")
print(data)
print("\nScaled data (mean≈0, std≈1 per column):")
print(data_scaled.round(3))

Learned mean: [ 25. 250.]
Learned std:  [ 11.18033989 111.80339887]

Original data:
[[ 10 200]
 [ 20 400]
 [ 30 100]
 [ 40 300]]

Scaled data (mean≈0, std≈1 per column):
[[-1.342 -0.447]
 [-0.447  1.342]
 [ 0.447 -1.342]
 [ 1.342  0.447]]


In [19]:
# ── Data leakage demo: why we only fit on train data ──────────────────
X_raw = np.array([[1], [2], [3], [4], [100]])   # 100 is an outlier in test
X_tr, X_te = X_raw[:3], X_raw[3:]

sc = StandardScaler()
sc.fit(X_tr)                    # Only uses train data!
print("Train scaled:", sc.transform(X_tr).flatten().round(2))
print("Test scaled: ", sc.transform(X_te).flatten().round(2))  # uses SAME stats

Train scaled: [-1.22  0.    1.22]
Test scaled:  [  2.45 120.02]


In [ ]:
# ── fit_transform() is a convenient shorthand ─────────────────────────
from sklearn.preprocessing import StandardScaler
sc2 = StandardScaler()

# These two lines…
# sc2.fit(data)
# result = sc2.transform(data)

# …are equivalent to this one:
result = sc2.fit_transform(data)
print("fit_transform result:")
print(result.round(3))

---
## 6. Common Preprocessing Transformers

### 6a. Encoding categorical variables

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

categories = np.array([["red"], ["green"], ["blue"], ["red"], ["green"]])

# OrdinalEncoder → integer per category (useful for tree models)
oe = OrdinalEncoder()
print("OrdinalEncoder output:")
print(oe.fit_transform(categories).flatten())
print("Categories learned:", oe.categories_)

print()

# OneHotEncoder → binary dummy columns (for linear models / neural nets)
ohe = OneHotEncoder(sparse_output=False)
print("OneHotEncoder output:")
ohe_result = ohe.fit_transform(categories)
print(pd.DataFrame(ohe_result, columns=ohe.get_feature_names_out()))

### 6b. Handling missing values

In [ ]:
from sklearn.impute import SimpleImputer

X_missing = np.array([[1.0, 2.0],
                      [np.nan, 3.0],
                      [5.0, np.nan],
                      [7.0, 6.0]])

# strategy='mean' fills NaN with column mean
imputer = SimpleImputer(strategy='mean')
X_filled = imputer.fit_transform(X_missing)

print("Original (with NaNs):")
print(X_missing)
print("\nAfter imputation (mean strategy):")
print(X_filled)
print("\nLearned statistics (means):", imputer.statistics_)

---
## 7. `Pipeline` – Chaining Steps Together

> A `Pipeline` bundles **preprocessing + model** into a single estimator.  
> It automatically calls `fit`/`transform` on each step and `fit`/`predict` on the final estimator.

```
Pipeline([('step_name', transformer_or_estimator), ...])
```

**Why use Pipeline?**
- Prevents data leakage (preprocessing is done correctly inside cross-validation)
- Cleaner, less error-prone code
- One `fit` call trains everything

In [20]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# ── Dataset ────────────────────────────────────────────────────────────
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ── Build the Pipeline ─────────────────────────────────────────────────
pipe = Pipeline([
    ('scaler', StandardScaler()),            # Step 1: scale features
    ('classifier', LogisticRegression(max_iter=1000))  # Step 2: classify
])

# ── Single fit call trains the whole pipeline ─────────────────────────
pipe.fit(X_train, y_train)

# ── predict goes through ALL steps automatically ──────────────────────
y_pred = pipe.predict(X_test)
print(f"Pipeline Accuracy: {accuracy_score(y_test, y_pred):.2%}")

Pipeline Accuracy: 97.37%


In [21]:
# ── Accessing individual steps after fitting ───────────────────────────
print("Scaler mean (first 5 features):")
print(pipe['scaler'].mean_[:5].round(3))
# OR: pipe.named_steps['scaler'].mean_

print("\nLogistic Regression coef (first 5):")
print(pipe['classifier'].coef_[0, :5].round(4))

Scaler mean (first 5 features):
[1.41180e+01 1.91850e+01 9.18820e+01 6.54378e+02 9.60000e-02]

Logistic Regression coef (first 5):
[-0.4319 -0.3873 -0.3934 -0.4652 -0.0717]


In [22]:
# ── transform up to (but not including) the last step ─────────────────
# pipe[:-1] gives you a sub-pipeline of all steps except the last
X_scaled = pipe[:-1].transform(X_test)
print(f"Shape after scaling: {X_scaled.shape}")
print(f"Mean of first column (should ≈ 0): {X_scaled[:, 0].mean():.3f}")

Shape after scaling: (114, 30)
Mean of first column (should ≈ 0): 0.014


### 7b. Pipeline with Imputation + Scaling + Model

In [23]:
from sklearn.impute import SimpleImputer

# Introduce some NaN values artificially
rng = np.random.default_rng(0)
X_noisy = X_train.copy().astype(float)
mask = rng.random(X_noisy.shape) < 0.05
X_noisy[mask] = np.nan
print(f"Missing values in training data: {np.isnan(X_noisy).sum()}")

# 3-step Pipeline: impute → scale → classify
pipe3 = Pipeline([
    ('imputer',    SimpleImputer(strategy='median')),
    ('scaler',     StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipe3.fit(X_noisy, y_train)

# Test data also might have NaN — pipeline handles it
X_test_noisy = X_test.copy().astype(float)
mask2 = rng.random(X_test_noisy.shape) < 0.05
X_test_noisy[mask2] = np.nan

y_pred3 = pipe3.predict(X_test_noisy)
print(f"Accuracy with 3-step Pipeline: {accuracy_score(y_test, y_pred3):.2%}")

Missing values in training data: 716
Accuracy with 3-step Pipeline: 97.37%


---
## 8. `ColumnTransformer` – Mixed-Type Data

> Real datasets have **both numeric and categorical** columns.  
> `ColumnTransformer` applies **different transformations** to different columns.

```python
ColumnTransformer([
    ('name', transformer, columns),
    ...
])
```
- `columns` can be a list of column names, indices, or a boolean mask
- Outputs are concatenated horizontally

In [24]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# ── Toy DataFrame with mixed types ────────────────────────────────────
df = pd.DataFrame({
    'age':    [25, 35, 45, 55, 30],
    'salary': [50000, 80000, 120000, 95000, 60000],
    'city':   ['NY', 'LA', 'NY', 'Chicago', 'LA'],
    'gender': ['M', 'F', 'M', 'F', 'M']
})
y_dummy = np.array([0, 1, 1, 1, 0])   # binary target

print(df)
print("\nDtypes:")
print(df.dtypes)

   age  salary     city gender
0   25   50000       NY      M
1   35   80000       LA      F
2   45  120000       NY      M
3   55   95000  Chicago      F
4   30   60000       LA      M

Dtypes:
age       int64
salary    int64
city        str
gender      str
dtype: object


In [25]:
# ── Define transformers for each column group ──────────────────────────
numeric_features     = ['age', 'salary']
categorical_features = ['city', 'gender']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(),              numeric_features),
    ('cat', OneHotEncoder(sparse_output=False), categorical_features)
])

# fit_transform on training data
X_processed = preprocessor.fit_transform(df)

# Show result as DataFrame with proper column names
cat_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)

result_df = pd.DataFrame(X_processed, columns=all_feature_names)
print("Transformed output:")
print(result_df.round(3))

Transformed output:
     age  salary  city_Chicago  city_LA  city_NY  gender_F  gender_M
0 -1.207  -1.241           0.0      0.0      1.0       0.0       1.0
1 -0.279  -0.040           0.0      1.0      0.0       1.0       0.0
2  0.650   1.561           0.0      0.0      1.0       0.0       1.0
3  1.578   0.560           1.0      0.0      0.0       1.0       0.0
4 -0.743  -0.841           0.0      1.0      0.0       0.0       1.0


### 8b. ColumnTransformer inside a Pipeline
This is the **most common production pattern**.

In [26]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# Larger dummy dataset
rng2 = np.random.default_rng(1)
n = 200
df2 = pd.DataFrame({
    'age':    rng2.integers(20, 70, n).astype(float),
    'salary': rng2.integers(30000, 150000, n).astype(float),
    'city':   rng2.choice(['NY', 'LA', 'Chicago', 'Houston'], n),
    'gender': rng2.choice(['M', 'F'], n)
})
# Binary target correlated with age and salary
y2 = ((df2['age'] > 40) & (df2['salary'] > 80000)).astype(int).values

df2_train, df2_test, y2_train, y2_test = train_test_split(
    df2, y2, test_size=0.25, random_state=0
)

# ── ColumnTransformer ──────────────────────────────────────────────────
ct = ColumnTransformer([
    ('num', StandardScaler(),                   ['age', 'salary']),
    ('cat', OneHotEncoder(sparse_output=False,
                          handle_unknown='ignore'), ['city', 'gender'])
])

# ── Full Pipeline ──────────────────────────────────────────────────────
full_pipe = Pipeline([
    ('preprocessor', ct),
    ('model', LogisticRegression(max_iter=500))
])

full_pipe.fit(df2_train, y2_train)
print(f"Accuracy: {accuracy_score(y2_test, full_pipe.predict(df2_test)):.2%}")

# Predict on a single new row — same pipeline, no extra preprocessing needed
new_person = pd.DataFrame({
    'age': [50], 'salary': [95000], 'city': ['LA'], 'gender': ['F']
})
print(f"\nPrediction for new person: {full_pipe.predict(new_person)[0]}")
print(f"Probability: {full_pipe.predict_proba(new_person)[0]}")

Accuracy: 94.00%

Prediction for new person: 1
Probability: [0.40250439 0.59749561]


---
## 9. Full End-to-End Example – Titanic Survival Prediction

We'll simulate a Titanic-like dataset and build a complete ML pipeline.

In [27]:
from sklearn.datasets import fetch_openml

# Load real Titanic dataset from OpenML
titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
df_titanic = titanic.data[['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()
y_titanic = (titanic.target == '1').astype(int)   # survived: 1, died: 0

print(f"Dataset shape: {df_titanic.shape}")
print(f"\nFirst 5 rows:")
print(df_titanic.head())
print(f"\nMissing values per column:")
print(df_titanic.isnull().sum())

Dataset shape: (1309, 7)

First 5 rows:
   pclass     sex      age  sibsp  parch      fare embarked
0       1  female  29.0000      0      0  211.3375        S
1       1    male   0.9167      1      2  151.5500        S
2       1  female   2.0000      1      2  151.5500        S
3       1    male  30.0000      1      2  151.5500        S
4       1  female  25.0000      1      2  151.5500        S

Missing values per column:
pclass        0
sex           0
age         263
sibsp         0
parch         0
fare          1
embarked      2
dtype: int64


In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, classification_report

# ── Column groups ──────────────────────────────────────────────────────
numeric_cols     = ['age', 'fare', 'sibsp', 'parch']
categorical_cols = ['pclass', 'sex', 'embarked']

# ── Sub-pipelines per type ─────────────────────────────────────────────
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# ── ColumnTransformer ──────────────────────────────────────────────────
preprocessor = ColumnTransformer([
    ('num', numeric_transformer,     numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

# ── Full Pipeline ──────────────────────────────────────────────────────
titanic_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   RandomForestClassifier(n_estimators=100, random_state=42))
])

# ── Train / test split ─────────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    df_titanic, y_titanic, test_size=0.2, random_state=42
)

titanic_pipe.fit(X_tr, y_tr)
y_pred_t = titanic_pipe.predict(X_te)

print(f"Test Accuracy: {accuracy_score(y_te, y_pred_t):.2%}")
print(classification_report(y_te, y_pred_t, target_names=['Died', 'Survived']))

Test Accuracy: 79.01%
              precision    recall  f1-score   support

        Died       0.77      0.88      0.82       144
    Survived       0.82      0.69      0.75       118

    accuracy                           0.79       262
   macro avg       0.80      0.78      0.78       262
weighted avg       0.79      0.79      0.79       262



---
## 10. Cross-Validation (the right way to evaluate)

Instead of a single train/test split, cross-validation gives a **more reliable estimate** of model performance.

In [29]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cross_val_score calls fit & predict automatically on each fold
scores = cross_val_score(
    titanic_pipe, df_titanic, y_titanic,
    cv=cv, scoring='accuracy'
)

print("5-Fold CV Accuracy scores:", scores.round(3))
print(f"Mean: {scores.mean():.3f} ± {scores.std():.3f}")

5-Fold CV Accuracy scores: [0.767 0.84  0.763 0.821 0.782]
Mean: 0.794 ± 0.030


---
## 11. Hyperparameter Tuning with `GridSearchCV`

> `GridSearchCV` searches over parameter combinations using cross-validation.  
> Parameter names in Pipeline use `stepname__param` syntax.

In [30]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'classifier__n_estimators': [50, 100],       # RF param
    'classifier__max_depth':    [None, 5, 10],   # RF param
}

grid_search = GridSearchCV(
    titanic_pipe, param_grid,
    cv=3, scoring='accuracy', n_jobs=-1
)
grid_search.fit(df_titanic, y_titanic)

print(f"Best params:   {grid_search.best_params_}")
print(f"Best CV score: {grid_search.best_score_:.3f}")

Best params:   {'classifier__max_depth': 10, 'classifier__n_estimators': 100}
Best CV score: 0.534


---
## 12. Quick Reference Summary

```python
# ── Core API ───────────────────────────────────────────────────
estimator.fit(X_train, y_train)       # Train (estimators & transformers)
estimator.predict(X_test)             # Predict labels
estimator.predict_proba(X_test)       # Predict probabilities
transformer.transform(X_test)         # Apply learned transformation
transformer.fit_transform(X_train)    # fit() + transform() in one step

# ── Pipeline ───────────────────────────────────────────────────
pipe = Pipeline([
    ('step1', SomeTransformer()),
    ('step2', AnotherTransformer()),
    ('model', SomeClassifier()),
])
pipe.fit(X_train, y_train)      # fits all steps in order
pipe.predict(X_test)            # transforms then predicts
pipe['step1']                   # access a step by name
pipe[:-1].transform(X_test)     # transform through all but last step

# ── ColumnTransformer ──────────────────────────────────────────
ct = ColumnTransformer([
    ('num', StandardScaler(),   numeric_cols),
    ('cat', OneHotEncoder(),    categorical_cols),
])
# Typically used inside a Pipeline as the preprocessor step

# ── Evaluation ─────────────────────────────────────────────────
cross_val_score(pipe, X, y, cv=5, scoring='accuracy')
GridSearchCV(pipe, param_grid, cv=5)
```

In [37]:
print(X_train)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
imputer.fit(X_train)        # learns the mean of each column from training data
X_filled = imputer.transform(X_test)   # replaces NaN with those learned means
X_filled

[[9.029e+00 1.733e+01 5.879e+01 ... 1.750e-01 4.228e-01 1.175e-01]
 [2.109e+01 2.657e+01 1.427e+02 ... 2.903e-01 4.098e-01 1.284e-01]
 [9.173e+00 1.386e+01 5.920e+01 ... 5.087e-02 3.282e-01 8.490e-02]
 ...
 [1.429e+01 1.682e+01 9.030e+01 ... 3.333e-02 2.458e-01 6.120e-02]
 [1.398e+01 1.962e+01 9.112e+01 ... 1.827e-01 3.179e-01 1.055e-01]
 [1.218e+01 2.052e+01 7.722e+01 ... 7.431e-02 2.694e-01 6.878e-02]]


array([[1.247e+01, 1.860e+01, 8.109e+01, ..., 1.015e-01, 3.014e-01,
        8.750e-02],
       [1.894e+01, 2.131e+01, 1.236e+02, ..., 1.789e-01, 2.551e-01,
        6.589e-02],
       [1.546e+01, 1.948e+01, 1.017e+02, ..., 1.514e-01, 2.837e-01,
        8.019e-02],
       ...,
       [1.152e+01, 1.493e+01, 7.387e+01, ..., 9.608e-02, 2.664e-01,
        7.809e-02],
       [1.422e+01, 2.785e+01, 9.255e+01, ..., 8.219e-02, 1.890e-01,
        7.796e-02],
       [2.073e+01, 3.112e+01, 1.357e+02, ..., 1.659e-01, 2.868e-01,
        8.218e-02]], shape=(114, 30))